In [1]:
import sys

def generate_cst_res(pdb_file, target_residues, anchor_res, output):
    """
    target_residues: lista de inteiros com os resíduos específicos (numeração Rosetta/PDB)
    anchor_res: resíduo âncora (ex: um resíduo fixo ou o primeiro da região)
    """
    coords = {}
    
    # 1. Ler as coordenadas CA do arquivo PDB
    with open(pdb_file) as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                res_num = int(line[22:26].strip())
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords[res_num] = (x, y, z)

    # Validar se o resíduo âncora existe no PDB
    if anchor_res not in coords:
        print(f"Erro: O resíduo âncora {anchor_res} não foi encontrado no PDB.")
        return

    # 2. Gerar as constraints apenas para os resíduos selecionados
    constraints_count = 0
    with open(output, "w") as out:
        for res in target_residues:
            if res in coords:
                x, y, z = coords[res]
                out.write(
                    f"CoordinateConstraint CA {res} CA {anchor_res} "
                    f"{x:.3f} {y:.3f} {z:.3f} HARMONIC 0.0 1.0\n"
                )
                constraints_count += 1
            else:
                print(f"Aviso: Resíduo {res} especificado não foi encontrado no PDB e será pulado.")
                
    print(f"Gerado: {output} com {constraints_count} constraints criadas.")

In [4]:
target_residues = [72, 73,
                    186, 234, 242,
                    245, 246, 249,
                    250, 253, 270,
                    281, 282, 285
                    ]
input_file = "target.BL00010001.pdb"

In [5]:
# Constrain catalytic site
generate_cst_res(input_file, target_residues, anchor_res=target_residues[0], output="cst.cst")

Gerado: cst.cst com 14 constraints criadas.
